<h1>Data Preprocessing</h1>

This dataset contains household level transactions over two years from a group of 2,500 households who are frequent shoppers at a retailer. It contains all of each household’s purchases, not just those from a limited number of categories. For certain households, demographic information as well as direct marketing contact history are included.

Due to the number of tables and the overall complexity of The Complete Journey, it is suggested that this database be used in more advanced classroom settings. Further, The Complete Journey would be ideal for academic research as it should enable one to study the effects of direct marketing to customers.

In [2]:
import pandas as pd
import numpy as np

In [3]:
demo = pd.read_csv('./data/hh_demographic.csv')
prod = pd.read_csv('./data/product.csv')
trans = pd.read_csv('./data/transaction_data.csv')

demo.head()

,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


In [4]:
prod.head()

,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [5]:
trans.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


Summary of tables: 
1. HH Demographic shows the demographic of purchasers. It shows the following characteristics 
     - age bucket 
     - marital status 
     - income bucket 
     - homeowner status
     - dependent status or how many people in household 
     - household size 
     - kid category 
     - household key (identifier)
1. Product shows the products and their information 
     - product_id (identifier)
     - manufacturer 
     - department 
     - brand 
     - commodity description 
     - sub commodity description 
     - current size of product
1. Transactions shows the transaction information  
     - household key (identifying party)
     - basket_id 
     - day 
     - product id 
     - quantity 
     - sales value 
     - store_id 
     - retail_discount 
     - transaction time 
     - week_no 

### exploring slicing patterns 

In [44]:
prod['DEPARTMENT'].unique()

<StringArray>
[        'GROCERY',    'MISC. TRANS.',          'PASTRY',         'DRUG GM',
      'MEAT-PCKGD',   'SEAFOOD-PCKGD',         'PRODUCE',       'NUTRITION',
            'DELI',       'COSMETICS',            'MEAT',          'FLORAL',
 'TRAVEL & LEISUR',         'SEAFOOD', 'MISC SALES TRAN',       'SALAD BAR',
       'KIOSK-GAS', 'ELECT &PLUMBING',      'GRO BAKERY',    'GM MERCH EXP',
  'FROZEN GROCERY',  'COUP/STR & MFG',         'SPIRITS',   'GARDEN CENTER',
            'TOYS', 'CHARITABLE CONT',      'RESTAURANT',              'RX',
  'PROD-WHS SALES',       'MEAT-WHSE',      'DAIRY DELI',     'CHEF SHOPPE',
             'HBC',  'DELI/SNACK BAR',            'PORK',      'AUTOMOTIVE',
    'VIDEO RENTAL',               ' ', 'CNTRL/STORE SUP',      'HOUSEWARES',
   'POSTAL CENTER',           'PHOTO',           'VIDEO', 'PHARMACY SUPPLY']
Length: 44, dtype: str

In [11]:
#group the number of occurences of each commodity description then count 
prod_desc = trans.merge(prod, on="PRODUCT_ID", how='inner') 
prod_desc.groupby('COMMODITY_DESC')['BASKET_ID'].count().reset_index().sort_values(by="BASKET_ID", ascending=False).head(50)

,COMMODITY_DESC,BASKET_ID
273,SOFT DRINKS,117532
124,FLUID MILK PRODUCTS,85630
15,BAKED BREAD/BUNS/ROLLS,83232
52,CHEESE,74885
14,BAG SNACKS,67190
140,FRZN MEAT/MEAT DINNERS,56064
25,BEEF,48726
274,SOUP,46135
307,YOGURT,44697
134,FROZEN PIZZA,43362


Choose to use the cold cereal category

### clean data and labels 

In [12]:
prod.isna().sum()

PRODUCT_ID              0
MANUFACTURER            0
DEPARTMENT              0
BRAND                   0
COMMODITY_DESC          0
SUB_COMMODITY_DESC      0
CURR_SIZE_OF_PRODUCT    0
dtype: int64

first clean the names to make lowercase

In [13]:
for col in prod.columns: 
    current_col = col 
    lower_col = current_col.lower()
    prod = prod.rename(columns={current_col:lower_col})
    
prod = prod.drop(columns=['sub_commodity_desc', 'curr_size_of_product'])
prod.to_csv('./data/product_clean.csv', index=False)


In [14]:
trans.columns

Index(['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY',
       'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO',
       'COUPON_DISC', 'COUPON_MATCH_DISC'],
      dtype='str')

In [15]:
trans.isna().sum()

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64

In [16]:
for col in trans.columns:
    current_col = col 
    lower_col = current_col.lower()
    trans = trans.rename(columns={current_col:lower_col})
drop_cols = ['trans_time', 'store_id']
trans = trans.drop(columns=drop_cols)
trans.to_csv('./data/transaction_clean.csv', index=False)


### develop the indivdiual level data

Our source of heterogeniety is the household_key, as each individual we observe is going to have a household_key. Each transaction is a basket_id. We will then filter for a specific set of foods or categories: 



Individual level observations will look like the following: 

1. household id
1. num of purchases (n_x)
1. length of purchasing time (T)

but first, we have to filter transactions where the id is equal to that of cold cereal and then append the number of households who did not buy any cold cereal at all in this period

In [39]:
prod_clean = pd.read_csv('./data/product_clean.csv')
trans_clean = pd.read_csv('./data/transaction_clean.csv')
dmo = pd.read_csv('./data/hh_demographic.csv')

df = trans_clean.merge(prod_clean, on='product_id', how='inner')
filter_df = df[df['commodity_desc'] == 'COLD CEREAL'].reset_index(drop=True)

In [40]:
print("Customers who never bought cold cereal", abs(filter_df['household_key'].nunique() - df['household_key'].nunique()))

Customers who never bought cold cereal 358


In [41]:
print("Last day of study:",max(trans_clean['day']))
print("Last week of study:",max(trans_clean['week_no']))

Last day of study: 711
Last week of study: 102


In [42]:
filter_df = filter_df[['household_key', 'basket_id', 'day', 'week_no']]
filter_df_y1 = filter_df[filter_df['day'] <= 355]
filter_df_y2 = filter_df[filter_df['day'] > 355]

group_y1 = filter_df_y1.groupby('household_key').agg(
    total_baskets = ('basket_id', 'nunique'), 
    first_day = ('day', 'min'),
    first_week = ('week_no', 'min')
).reset_index() 
group_y1['repeat_baskets'] = group_y1['total_baskets'] - 1 
group_y1['individual_exposure_time_days'] = 355 - group_y1['first_day']
group_y1['individual_exposure_time_weeks'] = 51 - group_y1['first_week']
group_y1 = group_y1[(group_y1['individual_exposure_time_days'] > 0) & (group_y1['individual_exposure_time_weeks'] > 0)]
group_y1 = group_y1.drop(columns=['first_day', 'first_week'])

group_y2 = filter_df_y2.groupby('household_key').agg(
    total_baskets = ('basket_id', 'nunique'), 
    first_day = ('day', 'min'),
    first_week = ('week_no', 'min')
).reset_index() 
group_y2['repeat_baskets'] = group_y2['total_baskets'] - 1 
group_y2['individual_exposure_time_days'] = 711 - group_y2['first_day']
group_y2['individual_exposure_time_weeks'] = 102 - group_y2['first_week']
group_y2 = group_y2[(group_y2['individual_exposure_time_days'] > 0) & (group_y2['individual_exposure_time_weeks'] > 0)]
group_y2 = group_y2.drop(columns=['first_day', 'first_week'])


group_y2.head()

,household_key,total_baskets,repeat_baskets,individual_exposure_time_days,individual_exposure_time_weeks
0,1,8,7,336,48
1,3,2,1,345,49
2,6,3,2,247,35
3,7,10,9,319,45
4,8,9,8,344,49


In [43]:
group_y1.head()

,household_key,total_baskets,repeat_baskets,individual_exposure_time_days,individual_exposure_time_weeks
0,1,7,6,244,34
1,2,1,0,20,2
2,3,4,3,173,24
3,4,2,1,251,35
4,6,2,1,214,30


check that no new individuals are added or subtracted so that we track same individuals

In [44]:
matching_household_id_1 = group_y1['household_key'].isin(group_y2['household_key'])
matching_household_id_2 = group_y2['household_key'].isin(group_y1['household_key'])


y1_match = group_y1[matching_household_id_1]
y2_match = group_y2[matching_household_id_2]

In [45]:
print("y1 len", len(y1_match))
print("y2 len", len(y2_match))

y1 len 1564
y2 len 1564


Subtract 1 from each total baskets

In [46]:
y1_match.to_csv('./data/household_count_raw_y1.csv', index=False)
y2_match.to_csv('./data/household_count_raw_y2.csv', index=False)

In [22]:
#merge witgh demo data
dmo.head()

,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


In [23]:
dmo['KID_CATEGORY_DESC'].value_counts()

KID_CATEGORY_DESC
None/Unknown    558
1               114
3+               69
2                60
Name: count, dtype: int64

In [57]:
y1_merge = y1_match.merge(dmo[['household_key', 'KID_CATEGORY_DESC', 'INCOME_DESC']], on='household_key', how='inner') 
y2_merge = y2_match.merge(dmo[['household_key', 'KID_CATEGORY_DESC', 'INCOME_DESC']], on='household_key', how='inner')

y1_merge.head()


,household_key,total_baskets,repeat_baskets,individual_exposure_time_days,individual_exposure_time_weeks,KID_CATEGORY_DESC,INCOME_DESC
0,1,7,6,244,34,None/Unknown,35-49K
1,7,3,2,322,46,None/Unknown,50-74K
2,8,2,1,190,27,1,25-34K
3,13,5,4,176,25,2,75-99K
4,18,8,7,246,35,None/Unknown,100-124K


In [58]:
y1_merge['INCOME_DESC'].value_counts()

INCOME_DESC
50-74K       152
35-49K       136
75-99K        78
25-34K        61
15-24K        60
Under 15K     56
125-149K      32
100-124K      29
150-174K      27
250K+         10
175-199K       9
200-249K       5
Name: count, dtype: int64

In [59]:
def assign_income_level(cat): 
    if cat in ['Under 15K', '15-24K', '25-34K', '35-49K']: 
        return 'low'
    elif cat in ['50-74K', '75-99K']: 
        return 'mid' 
    return 'high'

y1_merge['income_level'] = y1_merge['INCOME_DESC'].map(assign_income_level)


In [60]:
y1_merge

,household_key,total_baskets,repeat_baskets,individual_exposure_time_days,individual_exposure_time_weeks,KID_CATEGORY_DESC,INCOME_DESC,income_level
0,1,7,6,244,34,None/Unknown,35-49K,low
1,7,3,2,322,46,None/Unknown,50-74K,mid
2,8,2,1,190,27,1,25-34K,low
3,13,5,4,176,25,2,75-99K,mid
4,18,8,7,246,35,None/Unknown,100-124K,high
...,...,...,...,...,...,...,...,...
650,2494,1,0,239,34,None/Unknown,50-74K,mid
651,2496,2,1,190,27,1,75-99K,mid
652,2497,1,0,208,29,None/Unknown,35-49K,low
653,2498,4,3,79,11,None/Unknown,50-74K,mid


In [61]:
y1_merge

,household_key,total_baskets,repeat_baskets,individual_exposure_time_days,individual_exposure_time_weeks,KID_CATEGORY_DESC,INCOME_DESC,income_level
0,1,7,6,244,34,None/Unknown,35-49K,low
1,7,3,2,322,46,None/Unknown,50-74K,mid
2,8,2,1,190,27,1,25-34K,low
3,13,5,4,176,25,2,75-99K,mid
4,18,8,7,246,35,None/Unknown,100-124K,high
...,...,...,...,...,...,...,...,...
650,2494,1,0,239,34,None/Unknown,50-74K,mid
651,2496,2,1,190,27,1,75-99K,mid
652,2497,1,0,208,29,None/Unknown,35-49K,low
653,2498,4,3,79,11,None/Unknown,50-74K,mid


In [62]:
y1_merge = y1_merge.drop(columns=['INCOME_DESC'])

In [71]:
y1_merge['total_baskets'].mean()

np.float64(8.587786259541986)

In [63]:
y1_kids = y1_merge[y1_merge['KID_CATEGORY_DESC'].isin(['1', '3+', '2'])]
y1_no_kids = y1_merge[y1_merge['KID_CATEGORY_DESC'] == 'None/Unknown'] 
y2_kids = y2_merge[y2_merge['KID_CATEGORY_DESC'].isin(['1', '3+', '2'])]
y2_no_kids = y2_merge[y2_merge['KID_CATEGORY_DESC'] == 'None/Unknown'] 

In [64]:
print("Y1 With kids:", len(y1_kids))
print("Y1 with no kids:", len(y1_no_kids))
print("Y2 With kids:", len(y2_kids))
print("Y2 with no kids:", len(y2_no_kids))

Y1 With kids: 210
Y1 with no kids: 445
Y2 With kids: 210
Y2 with no kids: 445


In [65]:
y1_merge.to_csv('./data/household_count_raw_y1.csv', index=False) 
y2_merge.to_csv('./data/household_count_raw_y2.csv', index=False)

In [38]:
y1_no_kids.to_csv('./data/household_count_raw_y1_no_kids.csv', index=False)
y1_kids.to_csv('./data/household_count_raw_y1_kids.csv', index=False)
y2_no_kids.to_csv('./data/household_count_raw_y2_no_kids.csv', index=False)
y2_kids.to_csv('./data/household_count_raw_y2_kids.csv', index=False)

Now segment by income levels 

In [66]:
y1_low = y1_merge[y1_merge['income_level'] == 'low']
y1_mid = y1_merge[y1_merge['income_level'] == 'mid']
y1_high = y1_merge[y1_merge['income_level'] == 'high']

y1_low.to_csv('./data/subgroups/income/y1_low_income.csv', index=False)
y1_mid.to_csv('./data/subgroups/income/y1_mid_income.csv', index=False)
y1_high.to_csv('./data/subgroups/income/y1_high_income.csv', index=False)


In [67]:
y1_mid

,household_key,total_baskets,repeat_baskets,individual_exposure_time_days,individual_exposure_time_weeks,KID_CATEGORY_DESC,income_level
1,7,3,2,322,46,None/Unknown,mid
3,13,5,4,176,25,2,mid
6,20,37,36,274,39,None/Unknown,mid
7,22,29,28,330,47,None/Unknown,mid
8,25,2,1,232,33,None/Unknown,mid
...,...,...,...,...,...,...,...
647,2483,1,0,14,2,None/Unknown,mid
649,2489,13,12,256,36,None/Unknown,mid
650,2494,1,0,239,34,None/Unknown,mid
651,2496,2,1,190,27,1,mid
